In [1]:
import run as run
import chephy_model as cpm
import GNN_model
import gnn_utils
import numpy as np
import torch
import pandas as pd
import os


/home/dsi/chenbis/anaconda3/envs/chephy2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
atchley_path = "/home/dsi/chenbis/repos/sol_lab/files/atchley.csv"
atchley_df = pd.read_csv(atchley_path)
atchley_dict = atchley_df.set_index("amino.acid").to_dict(orient="index")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

healthy_raw_path = ["/dsi/sbm/chen2/raw_data/healthy/TRB_h_v1/", "/dsi/sbm/chen2/raw_data/healthy/TRB_h_v2/"]
sick_raw_path = "/dsi/sbm/chen2/raw_data/sick/mixcr_4.7_TRB/"

healthy_processed_path = "/dsi/sbm/chen2/processed_data/healthy"
sick_processed_path = "/dsi/sbm/chen2/processed_data/sick"




In [3]:
from scipy.spatial.distance import squareform # don't forget to use!!


def create_hetero_data(data, cdr3_header, distance_threshold = 0.2, use_mst=True, enrich=False):

    # data = data.sample(n=n, random_state=42)

    data = run.prepare_data(data, cdr3_header, epitope_header="")    
    data = cpm.truncate_sequences(data, cdr3_header, 4, 4)

    sequences = set(data["cdr3_truncated"])
    # epitopes = list(set(data[label_header]))

    chephy_matrix, sequences_list = cpm.find_che_phy_dist(sequences)
    sequence_indices = np.arange(len(sequences_list))

    data['tcr_index'] = data['cdr3_truncated'].apply(lambda x: sequences_list.index(x) if x in sequences_list else -1)
    # data['epitope_index'] = data[label_header].apply(lambda x: epitopes.index(x) if x in epitopes else -1)
    # print("Done indexing")

    data_hetero = gnn_utils.build_hetero_graph_cancer(
                            # data,
                            chephy_matrix,
                            sequences_list,
                            atchley_dict,
                            distance_threshold=distance_threshold,
                            use_mst=use_mst,
                            enrich=enrich  
                        )
    return data_hetero, data


In [4]:
def get_hetero_data(path, cdr3_header, distance_threshold=0.05, use_mst=False, enrich=True):

    df = pd.read_csv(path, sep="\t")
    # cdr3_header = 

    distance_threshold=0.05
    use_mst=False
    enrich=True

    data_hetero_train = create_hetero_data(df, cdr3_header, distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)    
    return data_hetero_train

In [5]:
from pathlib import Path
file_names_sick = [sick_raw_path + f.name for f in Path(sick_raw_path).iterdir() if f.is_file()]
file_names_healthy = [healthy_raw_path[0] + f.name for f in Path(healthy_raw_path[0]).iterdir() if f.is_file()]
file_names_healthy += [healthy_raw_path[1] + f.name for f in Path(healthy_raw_path[1]).iterdir() if f.is_file()]


In [6]:
def get_matrix(data, cdr3_header):

    data = run.prepare_data(data, cdr3_header, epitope_header="")    
    data = cpm.truncate_sequences(data, cdr3_header, 4, 4)

    sequences = set(data["cdr3_truncated"])

    chephy_matrix, sequences_list = cpm.find_che_phy_dist(sequences)
    # data['tcr_index'] = data['cdr3_truncated'].apply(lambda x: sequences_list.index(x) if x in sequences_list else -1)

    return chephy_matrix, sequences_list

In [8]:
from tqdm import tqdm
cdr3_header = "aaSeqCDR3"

for file_name in tqdm(file_names_sick, desc="Processing sick files matrices"):
    df = pd.read_csv(file_name, sep="\t")
    matrix, sequences_list = get_matrix(df, cdr3_header)
    np.savez_compressed(f'{sick_processed_path}/{file_name.rsplit("/",1)[-1]}.npz',
                    matrix=matrix,
                    sequences_list=sequences_list)


for file_name in tqdm(file_names_healthy, desc="Processing healthy files matrices"):
    df = pd.read_csv(file_name, sep="\t")
    matrix, sequences_list = get_matrix(df, cdr3_header)
    np.savez_compressed(f'{healthy_processed_path}/{file_name.rsplit("/",1)[-1]}.npz',
                    matrix=matrix,
                    sequences_list=sequences_list)   
    
    


Processing sick files matrices:  13%|█▎        | 6/47 [18:45<3:49:49, 336.34s/it]/tmp/ipykernel_371495/3864891194.py:5: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_name, sep="\t")
Processing sick files matrices:  26%|██▌       | 12/47 [25:53<41:22, 70.92s/it]   /tmp/ipykernel_371495/3864891194.py:5: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_name, sep="\t")
Processing healthy files matrices:  45%|████▌     | 23/51 [51:36<57:29, 123.21s/it]  /tmp/ipykernel_371495/3864891194.py:13: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_name, sep="\t")
Processing healthy files matrices:  65%|██████▍   | 33/51 [2:20:38<1:38:00, 326.69s/it] /tmp/ipykernel_371495/3864891194.py:13: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set 

In [ ]:

# Replace with the correct path to your .npz file
file_path = f"{healthy_processed_path}/{file_name.rsplit('/', 1)[-1]}.npz"

# Load the .npz file
data = np.load(file_path, allow_pickle=True)

# Access the contents
matrix = data['matrix']
sequences_list = data['sequences_list']

In [ ]:
from tqdm import tqdm
cdr3_header = "aaSeqCDR3"

distance_threshold=0.05
use_mst=False
enrich=True
sick_hetero_data = []
healthy_hetero_data = []


for file_name in tqdm(file_names_sick, desc="Processing sick files"):
    data_hetero = get_hetero_data(file_name, cdr3_header, distance_threshold=distance_threshold, use_mst=use_mst, enrich=enrich)
    sick_hetero_data.append(data_hetero)

torch.save(sick_hetero_data, file_name + "_hetero.pt")

for file_name in tqdm(file_names_healthy, desc="Processing healthy files"):
    data_hetero = get_hetero_data(file_name, cdr3_header, distance_threshold=distance_threshold, use_mst=use_mst, enrich=enrich)
    healthy_hetero_data.append(data_hetero)

torch.save(healthy_hetero_data, file_name + "_hetero.pt")